In [8]:
import os
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [9]:
documents = [
    "Ahmed is a third-year AI and Data Science student.",
    "He is learning Retrieval-Augmented Generation to build a dental lecture assistant.",
    "A convolutional neural network is commonly used for image classification tasks.",
    "Passive eruption is related to the movement of the gingival margin after tooth eruption.",
    "Power BI is used to build dashboards and visualize business data."
]

In [10]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
doc_embeddings = model.encode(documents)
print(type(doc_embeddings))
print(doc_embeddings.shape)

<class 'numpy.ndarray'>
(5, 384)


In [12]:
question = "What Project is Ahmed building with RAG"

question_embedding = model.encode(question)
print(question_embedding.shape)

(384,)


In [13]:
def cosine_similarity(a, b):
    dot_product = np.dot(a,b)
    mag_a = np.linalg.norm(a)
    mag_b = np.linalg.norm(b)
    return dot_product / (mag_a * mag_b)

In [14]:
similarties = []

for doc, emb in zip(documents, doc_embeddings):
    similarity = cosine_similarity(emb, question_embedding)
    similarties.append(similarity)

result = pd.DataFrame({"document": documents, "similarity": similarties})

result = result.sort_values(by="similarity", ascending=True)
result

,document,similarity
3,Passive eruption is related to the movement of...,-0.013756
2,A convolutional neural network is commonly use...,0.030239
1,He is learning Retrieval-Augmented Generation ...,0.148563
4,Power BI is used to build dashboards and visua...,0.192862
0,Ahmed is a third-year AI and Data Science stud...,0.365891


In [15]:
def retrieve_top_k(question, documents, doc_embeddings, k=3):
    question_embedding = model.encode(question)

    scores = []
    for emb in doc_embeddings:
        score = cosine_similarity(question_embedding, emb)
        scores.append(score)

    top_indices = np.argsort(scores)[::-1][:k]

    retrieved = []
    for idx in top_indices:
        retrieved.append({
            "document": documents[idx],
            "score": scores[idx]
        })

    return retrieved

In [16]:
retrieve_top_k(question, documents, doc_embeddings)

[{'document': 'Ahmed is a third-year AI and Data Science student.',
  'score': 0.36589068},
 {'document': 'Power BI is used to build dashboards and visualize business data.',
  'score': 0.19286232},
 {'document': 'He is learning Retrieval-Augmented Generation to build a dental lecture assistant.',
  'score': 0.14856318}]

In [17]:
question = "What is Ahmed Trying to build"

retrieved_chunks = retrieve_top_k(question=question,
                                  documents=documents,
                                  doc_embeddings=doc_embeddings,
                                  k=3)

for item in retrieved_chunks:
    print("Score: ", round(item["score"], 4))
    print("Chunk: ", item["document"])
    print("-" * 80)

Score:  0.5202
Chunk:  Ahmed is a third-year AI and Data Science student.
--------------------------------------------------------------------------------
Score:  0.1708
Chunk:  Power BI is used to build dashboards and visualize business data.
--------------------------------------------------------------------------------
Score:  0.1658
Chunk:  He is learning Retrieval-Augmented Generation to build a dental lecture assistant.
--------------------------------------------------------------------------------


In [18]:
def split_text_into_chunks(text, chunk_size=300, overlap=50):
    text = re.sub(r"/s+", " ", text)
    
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    
    return chunks

In [19]:
lecture_text = """
Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the teeth and covers the alveolar bone. Healthy gingiva is usually firm, pink, and stippled.

A convolutional neural network is a deep learning model commonly used for image classification. It uses convolutional layers to detect features such as edges, textures, and shapes.

Power BI is a business intelligence tool used to build dashboards and analyze business data.
"""

In [20]:
chunks = split_text_into_chunks(text=lecture_text)
for i, chunk in enumerate(chunks):
    print("Chunk ", i)
    print(chunk)
    print("-"  * 80)
    

Chunk  0

Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the oppos
--------------------------------------------------------------------------------
Chunk  1
ues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the tee
--------------------------------------------------------------------------------
Chunk  2
the part of the oral mucosa that surrounds the teeth and covers the alveolar bone. Healthy gingiva is usually firm, pink, and stippled.

A convolutional neural network is a deep learning model commonly used fo

In [21]:
chunk_embeddings = model.encode(chunks)

print(chunk_embeddings.shape)

(4, 384)


In [ ]:
question = "What is passive eruption?"

retrieved_chunks = retrieve_top_k(
    question=question,
    documents=chunks,
    doc_embeddings=chunk_embeddings,
    k=2
)

for item in retrieved_chunks:
    print("Score:", round(item["score"], 4))
    print("Text:", item["document"])
    print("-" * 80)


Score: 0.6646
Text: ues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the tee
--------------------------------------------------------------------------------
Score: 0.5718
Text: 
Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the oppos
--------------------------------------------------------------------------------


In [23]:
from pypdf import PdfReader

def extract_text_from_pdf(path):
    pdf = PdfReader(path)
    
    text_pages = []
    for page_number, page in enumerate(pdf.pages):
        text = page.extract_text()
        
        if text:
            text_pages.append({
                "page": page_number + 1,
                "text": text
            })
    
    return text_pages

In [24]:
pdf_path = "..\\data\\raw\\_10-K-2025-As-Filed.pdf"

pages = extract_text_from_pdf(pdf_path)

print("num of pages", len(pages))
print(pages[0]["page"])
print(pages[0]["text"][:1000])

num of pages 80
1
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐    TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California 94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
Cupertino, California 95014
(Address of principal executive offices) (Zip Code)
(408) 996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class
Trading 
symbol(s) Name of each exchange on which registered
Common Stock, $0.00001 par value per share AAPL The Nasdaq Stock M

In [25]:
def chunk_pdf_pages(pages, chunk_size=800, overlap=150):
    all_chunks = []
    for page in pages:
        page_number = page["page"]
        text = page["text"]
        
        chunks = split_text_into_chunks(text=text, chunk_size=chunk_size, overlap=overlap)
        
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page" : page_number,
                "chunk_id" : i,
                "text" : chunk
                }
            )
    return all_chunks


In [26]:
pdf_chunks = chunk_pdf_pages(
    pages,
    chunk_size=800,
    overlap=150
)

print("Number of chunks:", len(pdf_chunks))

print(pdf_chunks[0]["page"])
print(pdf_chunks[0]["chunk_id"])
print(pdf_chunks[0]["text"][:500])

Number of chunks: 467
1
0
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐    TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California 94-


In [27]:
pdf_text = [chunk["text"] for chunk in pdf_chunks]

pdf_embeddings = model.encode(pdf_text)

print(pdf_embeddings.shape)

(467, 384)


In [28]:
def retrieve_top_k_pdf(question, pdf_chunks, pdf_embeddings, k=3):
    question_embedding = model.encode(question)
    
    scores = []
    for emb in pdf_embeddings:
        score = cosine_similarity(emb, question_embedding)
        scores.append(score)
    
    top_indices = np.argsort(scores)[::-1][:k]
    
    retrieved = []
    for idx in top_indices:
        retrieved.append({
            "page": pdf_chunks[idx]["page"],
            "chunk_id": pdf_chunks[idx]["chunk_id"],
            "text": pdf_chunks[idx]["text"],
            "score": scores[idx]
            }
        )
    return retrieved

In [29]:
question = "What are the company's main risk factors?"

retrieved = retrieve_top_k_pdf(
    question=question,
    pdf_chunks=pdf_chunks,
    pdf_embeddings=pdf_embeddings,
    k=3
)

for item in retrieved:
    print("Page:", item["page"])
    print("Chunk:", item["chunk_id"])
    print("Score:", round(item["score"], 4))
    print(item["text"][:1000])
    print("-" * 100)

Page: 8
Chunk: 0
Score: 0.749
Item 1A. Risk Factors
The following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of 
operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these 
risks. Statements in this section are based on the Company’s beliefs and opinions regarding matters that could materially 
adversely affect the Company in the future and are not representations as to whether such matters have or have not occurred 
previously. The risks and uncertainties described below are not exhaustive and should not be considered a complete statement 
of all potential risks or uncertainties that the Company faces or may face in the future.
This section should be read in conjunction with Part II, Item
----------------------------------------------------------------------------------------------------
Page: 9
Chunk: 8
Score: 0.6208
o its customers, create 
delays and i

In [30]:
import faiss

embedding_dim = pdf_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

normalized_embeddings = pdf_embeddings.copy()
faiss.normalize_L2(normalized_embeddings)

index.add(normalized_embeddings)

print("vectors in index: ", index.ntotal)

vectors in index:  467


In [31]:
def retrieve_faiss(question, pdf_chunks, index, k=3):
    question_embedding = model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")
    
    faiss.normalize_L2(question_embedding)
    
    scores, indices = index.search(question_embedding, k)
    
    retrieved = []
    
    for idx, score in zip(indices[0], scores[0]):
        retrieved.append({
                    "page": pdf_chunks[idx]["page"],
                    "chunk_id": pdf_chunks[idx]["chunk_id"],
                    "text": pdf_chunks[idx]["text"],
                    "score": float(score)
                })
    
    return retrieved
    

In [32]:
question = "What are the company's main risk factors?"

retrieved = retrieve_faiss(question=question, pdf_chunks=pdf_chunks, index=index, k=3)

for item in retrieved:
    print("Page:", item["page"])
    print("Score:", round(item["score"], 4))
    print(item["text"][:1000])
    print("-" * 100)

Page: 8
Score: 0.749
Item 1A. Risk Factors
The following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of 
operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these 
risks. Statements in this section are based on the Company’s beliefs and opinions regarding matters that could materially 
adversely affect the Company in the future and are not representations as to whether such matters have or have not occurred 
previously. The risks and uncertainties described below are not exhaustive and should not be considered a complete statement 
of all potential risks or uncertainties that the Company faces or may face in the future.
This section should be read in conjunction with Part II, Item
----------------------------------------------------------------------------------------------------
Page: 9
Score: 0.6208
o its customers, create 
delays and inefficiencies in t

In [33]:
def build_prompt(question, retrieved):
    context_parts = []
    
    for idx, chunk in enumerate(retrieved, start=1):
        context_parts.append(f"[Source {idx} | Page {chunk['page']}]\n{chunk['text']}")
        
    context = '/n/n'.join(context_parts)
    
    prompt = f"""
You are a helpful study assistant.

Answer the question using ONLY the provided context.

If the answer is not found in the context, say:
"I could not find the answer in the provided document."

Question:
{question}

Context:
{context}

Answer:
"""

    return prompt

In [34]:
question = "What are the company's main risk factors?"

retrieved = retrieve_faiss(question=question, pdf_chunks=pdf_chunks, index=index, k=3)

prompt = build_prompt(question=question, retrieved=retrieved)

print(prompt)


You are a helpful study assistant.

Answer the question using ONLY the provided context.

If the answer is not found in the context, say:
"I could not find the answer in the provided document."

Question:
What are the company's main risk factors?

Context:
[Source 1 | Page 8]
Item 1A. Risk Factors
The following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of 
operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these 
risks. Statements in this section are based on the Company’s beliefs and opinions regarding matters that could materially 
adversely affect the Company in the future and are not representations as to whether such matters have or have not occurred 
previously. The risks and uncertainties described below are not exhaustive and should not be considered a complete statement 
of all potential risks or uncertainties that the Company faces or may fac

In [35]:
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [36]:
def generate_answer(prompt):
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents=prompt
    )
    
    return response

In [37]:
def answer_question(question, pdf_chunks, index, k=3):
    retrieved = retrieve_faiss(
        question=question,
        pdf_chunks=pdf_chunks,
        index=index,
        k=k
    )

    prompt = build_prompt(question, retrieved)

    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved
    }

In [38]:
result = answer_question(
    question="What are the company's main risk factors?",
    pdf_chunks=pdf_chunks,
    index=index,
    k=3
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(f"Page {source['page']} | Score {source['score']:.4f}")
    print(source["text"][:500])
    print("-" * 80)

Question:
What are the company's main risk factors?

Answer:
sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""The company's main risk factors include:
* Adverse global and regional economic conditions.
* Risks of industrial accidents at its suppliers and contract manufacturers.
* Major public health issues, including pandemics such as the COVID-19 pandemic."""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='HPESapqUGY3gnsEPnpfQMA' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=49,
  prompt_token_count=580,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=580
    ),
  ],
  thoughts_token_count=644,
  total_token_count=1273
) model_status=None automatic_function_calling_histo